In [ ]:
import warnings

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)


# Function definitions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"


def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break

    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'

    return result, duration_str


def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    # Apply entry time offset to the signal time
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None

    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (
                1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)

    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None


def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None, percentage_change=None,
                    time_limit_minutes=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])

    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']

        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'

        signal_open_price = price_data.at[signal_datetime + pd.Timedelta(minutes=entry_time_offset), 'Open']

        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)

        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if result == 1:
            current_margin = current_margin * (1 + tp)
        elif result == -1:
            current_margin = current_margin * (1 - sl)

        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin

        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])

        output_data = pd.concat([output_data, new_row], ignore_index=True)

    return output_data


import pandas as pd

# Load the new datasets
price_data = pd.read_csv('E:\Signal Backtesting\Input\Price_2024.csv', parse_dates=['Datetime'], index_col='Datetime')
signal_data = pd.read_csv('E:\Signal Backtesting\Input\Signature_AI_Results_Final.csv', parse_dates=['Datetime'])


In [14]:
import numpy as np
import pandas as pd

# Assume the trading strategy and its auxiliary functions are already defined in this script

# Define the parameter ranges
tp_values = np.arange(0.005, 0.02, 0.0001)
sl_values = np.arange(0.005, 0.02, 0.0001)
entry_time_offset_values = np.arange(0, 240, 1)
percentage_change_values = np.arange(0, 0.01, 0.0001)

# Genetic algorithm parameters
population_size = 100
num_generations = 10
crossover_probability = 0.7
mutation_probability = 0.2
tournament_size = 3

# Create the initial population
def create_individual():
    return [
        np.random.choice(tp_values),
        np.random.choice(sl_values),
        np.random.choice(entry_time_offset_values),
        np.random.choice(percentage_change_values)
    ]

def create_population(size):
    return [create_individual() for _ in range(size)]

# Evaluate the fitness of an individual
def evaluate(individual):
    tp, sl, entry_time_offset, percentage_change = individual
    
    # Filter data for the specific month
    month_price_data = price_data[price_data.index.month == specific_month]
    month_signal_data = signal_data[signal_data['Datetime'].dt.month == specific_month]
    
    # Run backtest with given parameters
    result = backtest_trades(month_price_data, month_signal_data, tp=tp, sl=sl, entry_time_offset=entry_time_offset, percentage_change=percentage_change, time_limit_minutes=120)
    
    # Calculate the final NAV
    final_nav = result['NAV'].iloc[-1]
    
    # Calculate ROI
    roi = ((final_nav - 100000) / 100000)*100
    
    return roi

# Tournament selection
def select(population, fitnesses, k):
    selected = []
    for _ in range(k):
        tournament = np.random.choice(len(population), tournament_size)
        tournament_fitnesses = [fitnesses[i] for i in tournament]
        winner = tournament[np.argmax(tournament_fitnesses)]
        selected.append(population[winner])
    return selected

# Crossover (Two-point crossover)
def crossover(parent1, parent2):
    if np.random.rand() < crossover_probability:
        point1, point2 = sorted(np.random.choice(len(parent1), 2, replace=False))
        child1 = parent1[:point1] + parent2[point1:point2] + parent1[point2:]
        child2 = parent2[:point1] + parent1[point1:point2] + parent2[point2:]
        return child1, child2
    else:
        return parent1, parent2

# Mutation (Uniform mutation)
def mutate(individual):
    if np.random.rand() < mutation_probability:
        index = np.random.choice(len(individual))
        if index == 0:
            individual[index] = np.random.choice(tp_values)
        elif index == 1:
            individual[index] = np.random.choice(sl_values)
        elif index == 2:
            individual[index] = np.random.choice(entry_time_offset_values)
        elif index == 3:
            individual[index] = np.random.choice(percentage_change_values)
    return individual

# Genetic algorithm
def genetic_algorithm():
    population = create_population(population_size)
    for generation in range(num_generations):
        fitnesses = [evaluate(ind) for ind in population]
        new_population = []
        
        # Selection
        selected = select(population, fitnesses, population_size // 2)
        
        # Crossover
        for i in range(0, len(selected), 2):
            parent1, parent2 = selected[i-1], selected[i]
            child1, child2 = crossover(parent1, parent2)
            new_population.extend([child1, child2])
        
        # Mutation
        new_population = [mutate(ind) for ind in new_population]
        
        population = new_population
    
    # Get the best individual
    fitnesses = [evaluate(ind) for ind in population]
    best_index = np.argmax(fitnesses)
    best_individual = population[best_index]
    best_tp, best_sl, best_entry_time_offset, best_percentage_change = best_individual

    # Calculate the optimized ROI
    month_price_data = price_data[price_data.index.month == specific_month]
    month_signal_data = signal_data[signal_data['Datetime'].dt.month == specific_month]
    result = backtest_trades(month_price_data, month_signal_data, tp=best_tp, sl=best_sl, entry_time_offset=best_entry_time_offset, percentage_change=best_percentage_change, time_limit_minutes=120)
    final_nav = result['NAV'].iloc[-1]
    optimized_roi = ((final_nav - 100000) / 100000)*100
    
    print(f"Month: {specific_month}")
    print(f"Best Take Profit: {best_tp}")
    print(f"Best Stop Loss: {best_sl}")
    print(f"Best Entry Time Offset: {best_entry_time_offset}")
    print(f"Best Percentage Change: {best_percentage_change}")
    print(f"Optimized ROI: {optimized_roi:.4f}")

# Function to optimize for each month
def optimize_for_month(month):
    global specific_month
    specific_month = month
    genetic_algorithm()

In [15]:
# Example: Optimize for January
optimize_for_month(1)

Month: 1
Best Take Profit: 0.01990000000000004
Best Stop Loss: 0.016900000000000033
Best Entry Time Offset: 158
Best Percentage Change: 0.0027
Optimized ROI: 23.7393


In [9]:
# Example: Optimize for January
optimize_for_month(2)

KeyError: Timestamp('2024-03-01 00:03:00')

In [16]:
# Example: Optimize for January
optimize_for_month(3)

Month: 3
Best Take Profit: 0.011300000000000017
Best Stop Loss: 0.019500000000000038
Best Entry Time Offset: 110
Best Percentage Change: 0.0057
Optimized ROI: 16.6979


In [17]:
# Example: Optimize for January
optimize_for_month(4)

Month: 4
Best Take Profit: 0.018600000000000037
Best Stop Loss: 0.019000000000000038
Best Entry Time Offset: 49
Best Percentage Change: 0.0005
Optimized ROI: 16.8901


In [12]:
# Example: Optimize for January
optimize_for_month(5)

Month: 5
Best Take Profit: 0.015300000000000029
Best Stop Loss: 0.011900000000000018
Best Entry Time Offset: 3
Best Percentage Change: 0.0023
Optimized ROI: 16.6025


In [13]:
# Example: Optimize for January
optimize_for_month(6)

Month: 6
Best Take Profit: 0.017900000000000034
Best Stop Loss: 0.017700000000000035
Best Entry Time Offset: 3
Best Percentage Change: 0.0059
Optimized ROI: 7.3169
